In [1]:
pip install openai neo4j pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import re
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from neo4j import GraphDatabase

In [3]:
load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
MODEL_NAME = os.getenv("OPENROUTER_MODEL", "google/gemma-4-31b-it:free")

NEO4J_URI = os.getenv("NEO4J_URI", "neo4j://127.0.0.1:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

print("MODEL_NAME:", MODEL_NAME)
print("NEO4J_URI:", NEO4J_URI)
print("NEO4J_USER:", NEO4J_USER)
print("NEO4J_DATABASE:", NEO4J_DATABASE)

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY belum diisi di file .env")

if not NEO4J_PASSWORD:
    raise ValueError("NEO4J_PASSWORD belum diisi di file .env")

MODEL_NAME: google/gemma-4-31b-it:free
NEO4J_URI: neo4j://127.0.0.1:7687
NEO4J_USER: neo4j
NEO4J_DATABASE: neo4j


In [4]:
# test openrouter 
llm_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

response = llm_client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": "Jawab singkat: apakah kamu bisa membuat query Cypher?"
        }
    ],
    temperature=0,
    max_tokens=100
)

print(response.choices[0].message.content)

Ya, saya bisa.


In [5]:
# test koneksi neo4j
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD)
)

driver.verify_connectivity()

print("Neo4j connection successful.")

Neo4j connection successful.


In [6]:
# fungsi run cypher
def run_cypher(query):
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query)
        records = [record.data() for record in result]

    return pd.DataFrame(records)

In [7]:
# test isi database
preview_query = """
MATCH (n)
RETURN labels(n)[0] AS label, count(n) AS total
ORDER BY total DESC
LIMIT 10
"""

preview = run_cypher(preview_query)
preview

,label,total
0,Occurrence,4264
1,Species,1542
2,Genus,1088
3,Family,426
4,Order,142
5,Class,31
6,Phylum,11
7,Year,7
8,Country,6
9,Kingdom,3


In [8]:
# shema graph untuk LLM
GRAPH_SCHEMA = """
You are an expert Neo4j Cypher assistant for a biodiversity knowledge graph.
Translate natural language questions into valid Neo4j Cypher queries.
DATABASE SCHEMA:
Node labels:
- Occurrence: occurrenceKey, decimalLatitude, decimalLongitude, source
- Species: name, pagerankScore, communityId, mlCluster, fastrp_embedding
- Genus: name
- Family: name
- Order: name
- Class: name
- Phylum: name
- Kingdom: name
- Country: name, pagerankScore
- Year: value, pagerankScore
- BasisOfRecord: name

Relationships:
- (:Occurrence)-[:OBSERVED_SPECIES]->(:Species)
- (:Occurrence)-[:RECORDED_IN]->(:Country)
- (:Occurrence)-[:RECORDED_IN_YEAR]->(:Year)
- (:Occurrence)-[:RECORDED_AS]->(:BasisOfRecord)
- (:Species)-[:BELONGS_TO]->(:Genus)
- (:Genus)-[:BELONGS_TO]->(:Family)
- (:Family)-[:BELONGS_TO]->(:Order)
- (:Order)-[:BELONGS_TO]->(:Class)
- (:Class)-[:BELONGS_TO]->(:Phylum)
- (:Phylum)-[:BELONGS_TO]->(:Kingdom)
- (:Species)-[:SIMILAR_TO_JACCARD {jaccardScore}]->(:Species)

Rules:
- Return ONLY the Cypher query.
- Do not explain.
- Do not use markdown.
- Use read-only Cypher only.
- Do not use CREATE, MERGE, DELETE, SET, REMOVE, DROP, LOAD CSV, APOC, or GDS.
- Always use LIMIT when returning rows.
"""

In [9]:
# cleaning output LLM
def clean_cypher(text):
    text = text.strip()
    text = re.sub(r"```cypher", "", text, flags=re.IGNORECASE)
    text = re.sub(r"```", "", text)
    text = text.strip()
    match = re.search(r"(MATCH|WITH|RETURN)\s", text, flags=re.IGNORECASE)
    if match:
        text = text[match.start():].strip()
    return text

In [10]:
# fungsi safety chechk
def is_safe_cypher(query):
    forbidden_keywords = [
        "CREATE",
        "MERGE",
        "DELETE",
        "DETACH",
        "SET",
        "REMOVE",
        "DROP",
        "LOAD CSV",
        "CALL GDS",
        "CALL APOC",
        "CALL DBMS"
    ]
    upper_query = query.upper()
    for keyword in forbidden_keywords:
        if keyword in upper_query:
            return False
    return True

In [11]:
# fungsi generate cypher
def generate_cypher(question):
    messages = [
        {
            "role": "system",
            "content": GRAPH_SCHEMA
        },
        {
            "role": "user",
            "content": f"""
Question:
{question}
Generate the Cypher query:
"""
        }
    ]
    response = llm_client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        temperature=0,
        max_tokens=700
    )
    cypher_query = response.choices[0].message.content
    cypher_query = clean_cypher(cypher_query)
    if not is_safe_cypher(cypher_query):
        raise ValueError(f"Generated query is not safe:\n{cypher_query}")
    return cypher_query

In [12]:
# test generate cypher 
question = "Negara mana yang memiliki jumlah occurrence paling banyak?"
cypher_query = generate_cypher(question)
print(cypher_query)

MATCH (c:Country)<-[:RECORDED_IN]-(o:Occurrence)
RETURN c.name, count(o) AS occurrenceCount
ORDER BY occurrenceCount DESC
LIMIT 1


In [13]:
# test jalanin cypher hasil LLM
result = run_cypher(cypher_query)
result

,c.name,occurrenceCount
0,Singapore,761


In [14]:
# fungsi lengkap ask graph
def ask_graph(question):
    print("QUESTION:")
    print(question)

    cypher_query = generate_cypher(question)

    print("\nGENERATED CYPHER:")
    print(cypher_query)

    print("\nQUERY RESULT:")
    result = run_cypher(cypher_query)

    return result

In [16]:
# demo 1
result_1 = ask_graph("Negara mana yang memiliki jumlah occurrence paling banyak?")
result_1

QUESTION:
Negara mana yang memiliki jumlah occurrence paling banyak?

GENERATED CYPHER:
MATCH (c:Country)<-[:RECORDED_IN]-(o:Occurrence)
RETURN c.name, count(o) AS occurrenceCount
ORDER BY occurrenceCount DESC
LIMIT 1

QUERY RESULT:


,c.name,occurrenceCount
0,Singapore,761


In [17]:
# demo 2
result_2 = ask_graph("Tahun apa yang memiliki jumlah occurrence paling tinggi?")
result_2

QUESTION:
Tahun apa yang memiliki jumlah occurrence paling tinggi?

GENERATED CYPHER:
MATCH (y:Year)<-[:RECORDED_IN_YEAR]-(o:Occurrence)
RETURN y.value AS Year, count(o) AS OccurrenceCount
ORDER BY OccurrenceCount DESC
LIMIT 1

QUERY RESULT:


,Year,OccurrenceCount
0,2025,644


In [18]:
# demo 3
result_3 = ask_graph("Species apa yang paling sering muncul dalam data occurrence?")
result_3

QUESTION:
Species apa yang paling sering muncul dalam data occurrence?

GENERATED CYPHER:
MATCH (s:Species)<-[:OBSERVED_SPECIES]-(o:Occurrence)
RETURN s.name, count(o) AS occurrenceCount
ORDER BY occurrenceCount DESC
LIMIT 10

QUERY RESULT:


,s.name,occurrenceCount
0,Lanius schach,58
1,Geopelia striata,48
2,Pycnonotus goiavier,44
3,Macaca fascicularis,42
4,Passer montanus,39
5,Columba livia,34
6,Lissachatina fulica,34
7,Oecophylla smaragdina,32
8,Neurothemis fluctuans,30
9,Hypolimnas bolina,27


In [19]:
# demo 4
result_4 = ask_graph("Family apa yang memiliki jumlah species paling banyak?")
result_4

QUESTION:
Family apa yang memiliki jumlah species paling banyak?

GENERATED CYPHER:
MATCH (s:Species)-[:BELONGS_TO]->(f:Family)
RETURN f.name, count(s) AS speciesCount
ORDER BY speciesCount DESC
LIMIT 1

QUERY RESULT:


""


In [20]:
# demo 5
result_5 = ask_graph("Tampilkan jumlah species pada setiap cluster K-Means.")
result_5

QUESTION:
Tampilkan jumlah species pada setiap cluster K-Means.

GENERATED CYPHER:
MATCH (s:Species)
RETURN s.mlCluster AS cluster, count(s) AS speciesCount
LIMIT 100

QUERY RESULT:


,cluster,speciesCount
0,2,385
1,1,660
2,0,497


In [21]:
# demo 6
result_6 = ask_graph("Tampilkan pasangan species dengan nilai Jaccard similarity tertinggi.")
result_6

QUESTION:
Tampilkan pasangan species dengan nilai Jaccard similarity tertinggi.

GENERATED CYPHER:
MATCH (s1:Species)-[r:SIMILAR_TO_JACCARD]->(s2:Species)
RETURN s1.name, s2.name, r.jaccardScore
ORDER BY r.jaccardScore DESC
LIMIT 1

QUERY RESULT:


,s1.name,s2.name,r.jaccardScore
0,Crepidium koordersii,Bryobium retusum,1.0


In [ ]:
# %%
driver.close()
print("Neo4j connection closed.")